In [ ]:
# first import the necessary libraries
# - typing: to define the types of the state
# - langchain_core.messages: to define the messages
# - langchain.schema: to define the schema of the documents
# - langgraph.graph: to define the graph
# - langchain_openai: to use the OpenAI API
# - langchain.prompts: to define the prompts
# - uuid: to generate unique ids

from typing import TypedDict, List, Dict, Any, Optional, Annotated
from langchain_core.messages import BaseMessage
from langchain.schema import Document
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
import uuid

In [ ]:
# here the this class is used to define the state of the graph
# it is a dictionary with the following keys:
# - documents: a list of documents
# - base_questions: a list of base questions
# - simple_evolved: a list of simple evolved questions
# - multicontext_evolved: a list of multicontext evolved questions
# - reasoning_evolved: a list of reasoning evolved questions
# - all_evolved_questions: a list of all evolved questions
# - question_answers: a list of question answers
# - question_contexts: a list of question contexts
# - messages: a list of messages

class SyntheticDataState(TypedDict):
    documents: List[Document]
    base_questions: List[Dict[str, Any]]
    simple_evolved: List[Dict[str, Any]]
    multicontext_evolved: List[Dict[str, Any]]
    reasoning_evolved: List[Dict[str, Any]]
    all_evolved_questions: List[Dict[str, Any]]
    question_answers: List[Dict[str, Any]]
    question_contexts: List[Dict[str, Any]]
    messages: List[BaseMessage]

In [ ]:
# here this function is used to create the base questions
# it takes the state as input and returns the state

def create_base_questions(state: SyntheticDataState) -> SyntheticDataState:

    base_question_prompt = ChatPromptTemplate.from_template("""
    Based on the following documents, generate 5 diverse questions that could be asked about this content.
    Focus on creating questions that cover different aspects: factual, procedural, and analytical.
    
    Documents:
    {documents}
    
    Generate questions in this format:
    1. [Question text]
    2. [Question text]
    ...
    """)
    
    llm = ChatOpenAI(model="gpt-4o-mini")
    chain = base_question_prompt | llm
    
    doc_content = "\n\n".join([doc.page_content for doc in state["documents"][:5]])
    
    response = chain.invoke({"documents": doc_content})
    
    # parse questions and create structured format
    questions = []
    for i, line in enumerate(response.content.split('\n')):
        if line.strip() and any(char.isdigit() for char in line):
            question_text = line.split('.', 1)[1].strip() if '.' in line else line.strip()
            question_id = f"base_{uuid.uuid4().hex[:8]}"
            questions.append({
                "id": question_id,
                "question": question_text,
                "evolution_type": "base",
                "evolution_step": 0
            })
    
    return {
        **state,
        "base_questions": questions
    }

In [ ]:
# here we will create the simple evolved questions
# it is a function that performs simple evolution on questions

def simple_evolution_agent(state: SyntheticDataState) -> SyntheticDataState:
    
    simple_evolution_prompt = ChatPromptTemplate.from_template("""
    Take the following question and make it more specific, detailed, or clearer.
    The evolved question should be more precise but still answerable from the same context.
    
    Original Question: {question}
    
    Evolved Question:
    """)
    
    llm = ChatOpenAI(model="gpt-4o-mini")
    chain = simple_evolution_prompt | llm
    
    evolved_questions = []
    
    for question_data in state["base_questions"][:2]:  # Process first 2 questions
        response = chain.invoke({"question": question_data["question"]})
        evolved_question = response.content.strip()
        
        evolved_questions.append({
            "id": f"simple_{uuid.uuid4().hex[:8]}",
            "question": evolved_question,
            "evolution_type": "simple",
            "evolution_step": 1,
            "parent_id": question_data["id"]
        })
    
    return {
        **state,
        "simple_evolved": evolved_questions
    }


In [ ]:
# for the multicontext evolution we will use the following function
# creates a question that requires information from multiple documents to answer
# for example: "What is the capital of France and what is the population of Paris?"

def multi_context_evolution_agent(state: SyntheticDataState) -> SyntheticDataState:
    
    multi_context_prompt = ChatPromptTemplate.from_template("""
    Based on the following documents, create a question that requires information from multiple documents to answer.
    The question should be complex enough to need synthesis of information from different sources.
    
    Documents:
    {documents}
    
    Complex Multi-Context Question:
    """)
    
    llm = ChatOpenAI(model="gpt-4o-mini")
    chain = multi_context_prompt | llm
    
    doc_content = "\n\n".join([doc.page_content for doc in state["documents"][:3]])
    response = chain.invoke({"documents": doc_content})
    
    evolved_questions = [{
        "id": f"multicontext_{uuid.uuid4().hex[:8]}",
        "question": response.content.strip(),
        "evolution_type": "multi_context",
        "evolution_step": 1,
        "parent_id": None
    }]
    
    return {
        **state,
        "multicontext_evolved": evolved_questions
    }

In [ ]:
# for questions that requires multi-step reasoning
# takes base questions 1-3 and creates a question that requires multi-step reasoning

def reasoning_evolution_agent(state: SyntheticDataState) -> SyntheticDataState:
    """Agent that creates questions requiring multi-step reasoning."""
    
    reasoning_prompt = ChatPromptTemplate.from_template("""
    Create a question that requires multi-step reasoning and analysis.
    The question should require the user to:
    1. Understand multiple concepts
    2. Apply logic to connect different pieces of information
    3. Draw conclusions or make recommendations
    
    Base Question: {base_question}
    
    Complex Reasoning Question:
    """)
    
    llm = ChatOpenAI(model="gpt-4o-mini")
    chain = reasoning_prompt | llm
    
    evolved_questions = []
    
    for question_data in state["base_questions"][1:3]:  # Process 2 questions
        response = chain.invoke({"base_question": question_data["question"]})
        evolved_question = response.content.strip()
        
        evolved_questions.append({
            "id": f"reasoning_{uuid.uuid4().hex[:8]}",
            "question": evolved_question,
            "evolution_type": "reasoning",
            "evolution_step": 1,
            "parent_id": question_data["id"]
        })
    
    return {
        **state,
        "reasoning_evolved": evolved_questions
    }

In [ ]:
# here we are just combining the evolved questions
# we are adding the base questions, simple evolved questions, multicontext evolved questions, and reasoning evolved questions
# to the all_evolved_questions key in the state

def combine_evolved_questions(state: SyntheticDataState) -> SyntheticDataState:
    
    all_questions = (
        state["base_questions"] + 
        state["simple_evolved"] + 
        state["multicontext_evolved"] + 
        state["reasoning_evolved"]
    )
    
    return {
        **state,
        "all_evolved_questions": all_questions
    }

In [ ]:
# generating the answers for the evolved questions

def answer_generation_agent(state: SyntheticDataState) -> SyntheticDataState:
    
    answer_prompt = ChatPromptTemplate.from_template("""
    Based on the following documents, provide a comprehensive answer to the question.
    Use only information from the provided documents.
    
    Documents:
    {documents}
    
    Question: {question}
    
    Answer:
    """)
    
    llm = ChatOpenAI(model="gpt-4o-mini")
    chain = answer_prompt | llm
    
    doc_content = "\n\n".join([doc.page_content for doc in state["documents"][:5]])
    question_answers = []
    
    for question_data in state["all_evolved_questions"]:
        response = chain.invoke({
            "documents": doc_content,
            "question": question_data["question"]
        })
        
        question_answers.append({
            "question_id": question_data["id"],
            "answer": response.content.strip()
        })
    
    return {
        **state,
        "question_answers": question_answers
    }

In [ ]:
# here we are extracting the context for the questions
# the prompt is asking for the most relevant passages from the documents that would be needed to answer the question

def context_extraction_agent(state: SyntheticDataState) -> SyntheticDataState:
    
    context_prompt = ChatPromptTemplate.from_template("""
    For the given question, identify the most relevant passages from the documents that would be needed to answer it.
    Extract 2-3 key passages that contain the essential information.
    
    Question: {question}
    
    Documents:
    {documents}
    
    Relevant Contexts (extract exact passages):
    """)
    
    llm = ChatOpenAI(model="gpt-4o-mini")
    chain = context_prompt | llm
    
    doc_content = "\n\n".join([doc.page_content for doc in state["documents"][:5]])
    question_contexts = []
    
    for question_data in state["all_evolved_questions"]:
        response = chain.invoke({
            "question": question_data["question"],
            "documents": doc_content
        })
        
        question_contexts.append({
            "question_id": question_data["id"],
            "contexts": response.content.strip()
        })
    
    return {
        **state,
        "question_contexts": question_contexts
    }

In [ ]:
# building the graph

def build_synthetic_data_graph() -> StateGraph:
    
    workflow = StateGraph(SyntheticDataState)
    
    workflow.add_node("create_base_questions", create_base_questions)
    workflow.add_node("simple_evolution", simple_evolution_agent)
    workflow.add_node("multi_context_evolution", multi_context_evolution_agent)
    workflow.add_node("reasoning_evolution", reasoning_evolution_agent)
    workflow.add_node("combine_questions", combine_evolved_questions)
    workflow.add_node("generate_answers", answer_generation_agent)
    workflow.add_node("extract_contexts", context_extraction_agent)
    
    workflow.set_entry_point("create_base_questions")
    
    workflow.add_edge("create_base_questions", "simple_evolution")
    workflow.add_edge("simple_evolution", "multi_context_evolution")
    workflow.add_edge("multi_context_evolution", "reasoning_evolution")
    workflow.add_edge("reasoning_evolution", "combine_questions")
    workflow.add_edge("combine_questions", "generate_answers")
    workflow.add_edge("generate_answers", "extract_contexts")
    workflow.add_edge("extract_contexts", END)
    
    return workflow.compile()

In [ ]:
# this generates the synthetic data
# 

def generate_synthetic_data_with_langgraph(documents: List[Document]) -> Dict[str, List[Dict]]:
    
    graph = build_synthetic_data_graph()
    
    initial_state = {
        "documents": documents,
        "base_questions": [],
        "simple_evolved": [],
        "multicontext_evolved": [],
        "reasoning_evolved": [],
        "all_evolved_questions": [],
        "question_answers": [],
        "question_contexts": [],
        "messages": []
    }
    
    final_state = graph.invoke(initial_state)
    
    evolved_questions = [
        {
            "id": q["id"],
            "question": q["question"],
            "evolution_type": q["evolution_type"]
        }
        for q in final_state["all_evolved_questions"]
    ]
    
    question_answers = final_state["question_answers"]
    question_contexts = final_state["question_contexts"]
    
    return {
        "evolved_questions": evolved_questions,
        "question_answers": question_answers,
        "question_contexts": question_contexts
    }

In [ ]:
import os
import getpass

# get the api keys from the user

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

In [ ]:
# load documents

from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

synthetic_data = generate_synthetic_data_with_langgraph(docs[:5])  # Limit for demo

In [ ]:
print("Evolved Questions:")
for q in synthetic_data["evolved_questions"]:
    print(f"ID: {q['id']}, Type: {q['evolution_type']}, Question: {q['question']}")

print("\nQuestion Answers:")
for qa in synthetic_data["question_answers"]:
    print(f"Question ID: {qa['question_id']}, Answer: {qa['answer'][:100]}...")

print("\nQuestion Contexts:")
for qc in synthetic_data["question_contexts"]:
    print(f"Question ID: {qc['question_id']}, Contexts: {qc['contexts'][:100]}...")

Evolved Questions:
ID: base_a8b79f55, Type: base, Question: What are the minimum requirements for weeks of instructional time in an academic year for programs offered in credit hours and clock hours according to the guidelines provided in Volume 3?
ID: base_367c44ce, Type: base, Question: How does the academic year definition affect the calculation of financial aid awards under Title IV programs for students enrolled in different types of academic calendars?
ID: base_ed0f113a, Type: base, Question: In what circumstances can a school request approval to establish an academic year of less than 30 weeks of instructional time for Title IV funding purposes?
ID: base_8da7677a, Type: base, Question: What modifications were made in the 2025-2026 volume regarding the cost of attendance for Pell Grant recipients enrolled in subscription-based programs?
ID: base_fedce7f3, Type: base, Question: Analyze how the distinction between standard term and non-term academic calendars might impact the disbu